# 🧹 Notebook 03 — Data Cleaning & QA/QC

**Project:** Texas Injection Wells — Delaware Basin Analysis  
**Author:** Juan David Antolinez  
**Purpose:** Clean and standardize the raw injection data scraped from the Texas RRC H10 portal. Apply quality filters, rename columns to match the industry standard schema, and calculate derived fields.

---

### Cleaning Steps
1. Convert `Month/Yr` to proper datetime format
2. Sort by API number and date
3. Remove rows where `BBLS` is null
4. Remove wells where total lifetime `BBLS` = 0 (never injected)
5. Drop duplicate API + month combinations
6. Rename columns to industry standard schema
7. Calculate derived fields: `InjectionUptime`, `SwdInjectionRate`, `GasInjectionRate`, `Type`
8. Reorder columns

### Data Quality Summary
| Step | Rows Before | Rows After | Removed |
|---|---|---|---|
| Raw input | 233,158 | — | — |
| Remove null BBLS | 233,158 | 220,521 | 12,637 |
| Remove zero-injection wells | 220,521 | 176,181 | 44,340 |
| Remove duplicates | 176,181 | 175,838 | 343 |
| **Final clean dataset** | — | **175,838** | — |

### Output
- `data/processed/resultados_h10_clean.xlsx` — clean, standardized dataset

## 1. Library Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Project Paths

In [2]:
BASE_DIR       = Path.cwd().parent
DATA_RAW       = BASE_DIR / "data" / "raw"       / "resultados_h10.xlsx"
DATA_PROCESSED = BASE_DIR / "data" / "processed" / "resultados_h10_clean.xlsx"

## 3. Load Raw Data

In [3]:
df = pd.read_excel(DATA_RAW)
print(f"Raw dataset: {len(df):,} rows × {df.shape[1]} columns")

## 4. Date Formatting & Sorting

Convert `Month/Yr` from string format (`'08/2025'`) to proper datetime (`2025-08-01`).  
Then sort by API number and date to establish chronological order per well.

In [4]:
# Convert to datetime — pandas defaults to day 1 of each month
df['Month/Yr'] = pd.to_datetime(df['Month/Yr'], format="%m/%Y")

# Sort chronologically per well
df = df.sort_values(by=['API No.', 'Month/Yr'])

print(f"Date range: {df['Month/Yr'].min().strftime('%b %Y')} → {df['Month/Yr'].max().strftime('%b %Y')}")

## 5. BBLS Quality Analysis

Understand the distribution of injection volume values before applying any filters.

In [5]:
print(f"Total rows:              {len(df):,}")
print(f"BBLS null:               {df['BBLS'].isna().sum():,}")
print(f"BBLS = 0:                {(df['BBLS'] == 0).sum():,}")
print(f"BBLS > 0:                {(df['BBLS'] > 0).sum():,}")
print(f"BBLS empty string:       {(df['BBLS'] == '').sum():,}")

## 6. Remove Null BBLS Rows

Drop rows where `BBLS` has no value — these are incomplete records with no injection data.

In [6]:
before = len(df)
df = df[df['BBLS'].notna()]
print(f"Removed {before - len(df):,} null BBLS rows → {len(df):,} rows remaining")

## 7. Remove Zero-Injection Wells

Remove wells where the **lifetime total of BBLS = 0** across all months.  
These wells were permitted but never injected — they add no analytical value.

> Note: Individual months with BBLS = 0 are kept — a well may be shut-in temporarily.

In [7]:
# Calculate total lifetime BBLS per API
bbls_per_api   = df.groupby('API No.')['BBLS'].sum()
apis_zero_bbls = bbls_per_api[bbls_per_api == 0].index

print(f"Wells with zero lifetime injection: {len(apis_zero_bbls):,}")

# Remove those wells
before = len(df)
df = df[~df['API No.'].isin(apis_zero_bbls)]
print(f"Removed {before - len(df):,} rows → {len(df):,} rows remaining")
print(f"Unique APIs remaining: {df['API No.'].nunique():,}")

## 8. Remove Duplicates

Drop duplicate API + month combinations — keep the first occurrence.

In [8]:
before = len(df)
df = df.drop_duplicates(subset=['API No.', 'Month/Yr'], keep='first')
print(f"Removed {before - len(df):,} duplicates → {len(df):,} rows remaining")

## 9. Rename Columns & Calculate Derived Fields

Rename columns to match the industry standard schema.  
Calculate derived fields: injection rates, uptime, well type, and basin.

In [9]:
# ── Rename columns ────────────────────────────────────────────────────────────
df = df.rename(columns={
    'API No.'                : 'WellApi',
    'UIC Number'             : 'UicNumber',
    'Due Date'               : 'DueDate',
    'Operator Name'          : 'OperatorName',
    'District'               : 'District',
    'Oil Lease No.'          : 'LeaseNo',
    'Field Name'             : 'FieldName',
    'Lease Name'             : 'LeaseName',
    'County'                 : 'County',
    'Well No.'               : 'WellNo',
    'Month/Yr'               : 'MonthYr',
    'Avg PSIG'               : 'AvgPsig',
    'Max PSIG'               : 'MaxPsig',
    'BBLS'                   : 'Bbls',
    'MCF'                    : 'Mcf',
    '# Readings'             : 'NumReadings',
    'Min PSIG AP'            : 'MinPsigAp',
    'Max PSIG AP'            : 'MaxPsigAp',
    '16. Interval From (ft)' : 'CompletedInjectionIntervalFrom',
    '16. Interval To (ft)'   : 'CompletedInjectionIntervalTo',
    '17. Tubing Packer (ft)' : 'DepthOfTubingPacker',
    '18. Fluids From Others' : 'InjectedFluidsOtherSources',
    '19. Injection Through'  : 'InjectionThrough',
    '20. Fluid Type'         : 'TypeOfFluidsInjected',
})

# ── Derived fields ────────────────────────────────────────────────────────────

# Days in month (used as injection uptime proxy)
df['InjectionUptime'] = df['MonthYr'].dt.days_in_month

# Basin identifier
df['Basin'] = 'Delaware Basin'

# Injection rates (volume per day)
df['SwdInjectionRate'] = df['Bbls'] / df['InjectionUptime']  # bbl/day
df['GasInjectionRate'] = df['Mcf']  / df['InjectionUptime']  # mcf/day

# Placeholder columns (populated in Notebook 04)
df['GisLatNad83']  = np.nan
df['GisLongNad83'] = np.nan
df['MidPerforation'] = np.nan
df['AvgBhp'] = np.nan
df['MaxBhp'] = np.nan
df['MinBhp'] = np.nan

# Well type classification
df['Type'] = 'SWD'  # default: Salt Water Disposal
df.loc[df['LeaseName'].str.contains('UNIT', na=False), 'Type'] = 'Waterflood'

print(f"Columns after renaming: {df.shape[1]}")

## 10. Reorder Columns

Organize columns into logical groups: well identifiers → technical specs → monthly data → geospatial → BHP.

In [10]:
column_order = [
    # ── Well identifiers ──────────────────────────────
    'WellApi', 'UicNumber', 'DueDate', 'OperatorName',
    'District', 'County', 'FieldName', 'LeaseName', 'LeaseNo', 'WellNo',
    # ── Technical specs ───────────────────────────────
    'CompletedInjectionIntervalFrom', 'CompletedInjectionIntervalTo',
    'MidPerforation', 'DepthOfTubingPacker',
    'InjectedFluidsOtherSources', 'InjectionThrough',
    'TypeOfFluidsInjected', 'Type', 'Basin',
    # ── Monthly injection data ─────────────────────────
    'MonthYr', 'Bbls', 'Mcf',
    'AvgPsig', 'MaxPsig', 'MinPsigAp', 'MaxPsigAp',
    'NumReadings', 'InjectionUptime',
    'SwdInjectionRate', 'GasInjectionRate',
    # ── Geospatial (populated in Notebook 04) ─────────
    'GisLatNad83', 'GisLongNad83',
    # ── BHP (not available in H10 data) ───────────────
    'AvgBhp', 'MaxBhp', 'MinBhp',
]

df = df[column_order]
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 11. Save Clean Dataset

In [11]:
df.to_excel(DATA_PROCESSED, index=False)
print(f"✅  Saved: {DATA_PROCESSED}")
print(f"    Rows:    {len(df):,}")
print(f"    Columns: {df.shape[1]}")
print(f"    Unique wells: {df['WellApi'].nunique():,}")